#  Calculation of features from available libraries

In [1]:
from Tools.DatasetTools.Commoms import *

In [2]:
sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')

dataset = 'Fe-Mo'  # 'Cr-Co-W' # 'Fe-Mo'#
components = dataset.split('-')
system=dataset.replace('-','')
from BopFoxFeaturizer.Featurizer import Featurizer

import Tools.DatasetTools.GeneralFeaturizer as gf

BS = pd.read_pickle(os.path.join(dataset, 'FullyCuratedParsedBriefSummary.pkl'))
AtomsObjects = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{system}-POSCAR-initial-rescaled-AtomsObjects.pkl')).dropna()
PymatgenStructures = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{system}-POSCAR-initial-rescaled-PymatgenStructures.pkl')).dropna()
SublatticeTags = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SUBLATICETAGS.pkl'))
SublatticeSorters = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SORTERS.pkl'))
SublatticeSorters.index = SublatticeSorters.index.str.strip()
SublatticeTags.index = SublatticeSorters.index.str.strip()

BS.dropna(inplace=True)

import numpy as np

# Prepare Extra features

In [3]:
from importlib.machinery import SourceFileLoader

In [4]:
from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
encoder = LabelEncoder()

In [5]:
Featurizer = SourceFileLoader('Featurizer', '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/BopFoxFeaturizer/Featurizer.py').load_module().Featurizer

In [6]:
Features = Featurizer(BS)

In [7]:
DatasetCompositionFeatures = Features.get_fractions_by_components()

In [8]:
#DatasetFeatures = pd.concat((DatasetCompositionFeatures, DatasetMagneticFeature, StructureNameFeature), axis=1)

##  Magnetism and structure

In [9]:
StructureNameFeature = BS.Phase
StructureNameFeature.name='Structure'
encoder.fit(StructureNameFeature)
DatasetStructureFeature = pd.Series(encoder.transform(StructureNameFeature), name='Structure', index = StructureNameFeature.index)

In [10]:
MagneticFeature = Features.MagFeature
MagneticFeature.name = 'Mag'
encoder.fit(MagneticFeature)
DatasetMagneticFeature = pd.Series(encoder.transform(MagneticFeature), name='Mag', index = StructureNameFeature.index)

In [11]:
DatasetFeatures = pd.concat([DatasetMagneticFeature, DatasetStructureFeature, DatasetCompositionFeatures, BS.num_atoms], axis = 1)

## Coordination Polyhedra feature

The first feature that we would like to have is the count of each CP in each sample. for that we construct a vector in the following way:

$$ N_{CN}^i = \#^i CP $$

Next feature we want is the composition in each CP. for this we choose to represent the elment numerically by their atomic numbers, and the CP-resolved composition becomes the average atomc numbers,

$$ Z_{CP} ^i = \dfrac{1}{n_{at}^i} \sum_{at \in CP} Z_{at} $$

In [12]:
SublatticeSorters

Fe-Mo/data/Fe_pv/POSCAR-initial/A15.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial                           [9, 10, 11, 12, 13, 14, 7, 8]
Fe-Mo/data/Fe_pv/POSCAR-initial/A15/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial                              [9, 10, 11, 12, 13, 14, 7, 8]
Fe-Mo/data/Fe_pv/POSCAR-initial/C14.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial           [15, 16, 17, 18, 7, 8, 9, 10, 11, 12, 13, 14]
Fe-Mo/data/Fe_pv/POSCAR-initial/C14/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial              [15, 16, 17, 18, 7, 8, 9, 10, 11, 12, 13, 14]
Fe-Mo/data/Fe_pv/POSCAR-initial/C15.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial                                   [7, 8, 9, 10, 11, 12]
                                                                                                                  ...                        
Fe-Mo/data/Mo_sv/POSCAR-initial/mu.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial        [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Fe-Mo/

In [13]:
SortingFeatures = gf.sorting_feature(AtomsObjects, SublatticeSorters, SublatticeTags)
SortingFeatures.sorters = gf.correct_sortings_fromphases(AtomsObjects, BS.Phase, SortingFeatures.sorters)
SortingFeatures.sublatticetags = gf.correct_occupation_fromphases(BS.Phase, SortingFeatures.sublatticetags, AtomsObjects.atoms)
sampleinspecial = BS.Phase.map(lambda p: p in gf.specialphases)
empty = SortingFeatures.sublatticetags.map(lambda sublat: '' in sublat)
SortingFeatures.sublatticetags[empty] = ['A']
wrong = SortingFeatures.sublatticetags.map(lambda sublat: 'A' not in sublat) 
fixable = SortingFeatures.sublatticetags.loc[wrong].map(type) == np.ndarray #.map(np.unique)
CNList = gf.get_sitecn(BS.Phase, AtomsObjects.atoms, SortingFeatures.sorters)

  0%|          | 0/252 [00:00<?, ?it/s]

## Position Features

In [14]:
elements = np.unique(BS.filter(regex='^atom_').values.ravel())
ABOCC = pd.concat([BS.filter(regex='atom_'), Features.occupation], axis = 1)
ABOCC.rename(columns={ABOCC.columns[-1]: 'index'}, inplace=True)

In [15]:
Positions = {}
for index, item in ABOCC.iterrows():
    if item['index'] == '':
        thisposition = {index: [item[f'atom_A']]*len(np.unique(gf.cn_dict[BS.Phase[index]]))}
    else:
        thisposition = {index: [item[f'atom_{occ}'] for occ in item['index'] ]}
    Positions.update(thisposition)
Positions = pd.DataFrame.from_dict(Positions, orient='index')
Positions[Positions.isnull()] = 0
for i, element in enumerate(elements):
    Positions[Positions==element] = i
Positions.columns = [f'Pos_{col+1}' for col in Positions.columns]
#Positions[Positions.Pos_1.map(type) == str] = np.nan

## Averages over Coordination polyhedra

### Number of each CP in each structure

In [16]:
CN = gf.featurize_series(CNList, CNList, normalization='NCP', return0 = False)
newcolumns = ['N'+col for col in CN.columns]
CN.columns = newcolumns

### Composition and volume of the CP

In [17]:
from mendeleev import element

In [18]:
AtomicNumbers=AtomsObjects.atoms.map(lambda a: a.numbers)
AtomicNumbers.name = 'AtomicNumbers'
symbols = dataset.split('-')
volums = {symb: element(symb).atomic_volume for symb in symbols}

In [19]:
AtomicVolumes = AtomsObjects.atoms.map(lambda a: [volums[at] for at in a.get_chemical_symbols()])

In [20]:
CPVol = gf.featurize_series(AtomicVolumes, CNList, return0=False, normalization='NCP')
newcolumns = ['V'+col for col in CPVol.columns]
CPVol.columns =  newcolumns

In [21]:
CPComp = gf.featurize_series(AtomicNumbers, CNList, return0=False, normalization='NCP')
newcolumns = ['Z'+col for col in CPComp.columns]
CPComp.columns = newcolumns

## Compile all the descriptors

In [22]:
DatasetFeatures = pd.concat([DatasetStructureFeature, DatasetMagneticFeature, DatasetCompositionFeatures, CN, CPVol, CPComp, BS.num_atoms, Positions], axis=1)
datasetfeatureslocation = os.path.join(dataset, 'Descriptors','DatasetFeatures.pkl')
CNListlocation = os.path.join(dataset, 'Descriptors', 'CNList.pkl')
DatasetFeatures.to_pickle(datasetfeatureslocation)
CNList.to_pickle(CNListlocation)

# Matminer Features 

In [23]:
from Tools.DatasetTools.GetPymatgenFeatures import *

KeyboardInterrupt: 

In [ ]:
descriptorslocation = os.path.join(dataset, 'Descriptors')
mmflatomic = os.path.join(descriptorslocation, 'matminer_atomic_features.pkl')
mmfdensity = os.path.join (descriptorslocation, 'matminer_density_features.pkl')
mmfcomposition =  os.path.join (descriptorslocation,'matminer_composition_features.pkl')
mmfstructure =  os.path.join (descriptorslocation,'matminer_structure_features.pkl')
mmsoapfeatures = os.path.join(descriptorslocation, 'matminer_soap_features.pkl')


BS['chemical_formula'] = get_chemical_formula(BS)

In [ ]:
BS['composition'] = StrToComposition().featurize_dataframe(BS, "chemical_formula")['composition']

In [ ]:
BS['atoms_objects'] = PymatgenStructures

In [ ]:
AtomicFeaturesMagpie = load_features(mmflatomic, BS, which='atomic')
DensitiFeatures= load_features(mmfdensity, BS, which='density')
CompositionFeatures = load_features(mmfcomposition, BS, which='composition')
# SOAP doesnt work from matminer
# StructureFeatures = load_features(mmfstructure, BS, which='structure')

In [ ]:
AtomicFeaturesMagpie.columns = AtomicFeaturesMagpie.columns.str.replace('MagpieData ','')
AtomicFeaturesMagpie.dropna(axis=1, inplace = True)
AtomicFeaturesMagpie.describe()

In [ ]:
DensitiFeatures.dropna(axis=1, inplace=True)
if DensitiFeatures.shape[1] > 0:
    DensitiFeatures.describe()

In [ ]:
CompositionFeatures.dropna(axis=1, inplace=True)
CompositionFeatures.describe()

# ACE Features 

In [ ]:
from importlib.machinery import SourceFileLoader

In [ ]:
from Tools.DatasetTools.ACEDescriptors import MyPyACECalculator 
MyPyACECalculator = SourceFileLoader('ACEer', 'Tools/DatasetTools/ACEDescriptors.py').load_module().MyPyACECalculator
default_options = SourceFileLoader('ACEer', 'Tools/DatasetTools/ACEDescriptors.py').load_module().default_options_dict

In [ ]:
ACEer = MyPyACECalculator(components = components)

In [ ]:
ACEFEATURES = AtomsObjects['atoms'].map(ACEer.get_ace_projections)
ACEFEATURES.name = 'ace_projections'

In [ ]:
expand_ace = gf.array_expansions(ACEFEATURES.to_frame(),['ace_projections']) 

In [ ]:
CNAV_ACE = gf.featurize_dataframe(expand_ace, CNList)

In [ ]:
CNAV_ACE

In [ ]:
selection = CNAV_ACE[Features.StrucNames == 'bcc'].filter(regex='_0$')

In [ ]:
selection

In [ ]:
fig, ax = plt.subplots()
ax.plot(selection.iloc[0].values)
ax.plot(selection.iloc[2].values)
ax.set_yscale('log')

In [ ]:
ACE_file = os.path.join(descriptorslocation, f'{dataset}-ACE-CNAV.pkl')

In [ ]:
CNAV_ACE.to_pickle(ACE_file)

## CHANGING functions

In [ ]:

import pyace

#pyace.create_multispecies_basis_config??

from pyace.basisextension import construct_bbasisconfiguration, create_multispace_basis_config
from pyace import BBasis

In [ ]:
new_multispace_config = copy.deepcopy(ACEer.multispace_basis_config)

In [ ]:
new_multispace_config.update(
    {'embeddings':
     {
        'Fe': {
            'npot': 'FinnisSinclairShiftedScaled',
            'fs_parameters': [1, 1], ## non-linear embedding function: 1*rho_1^1 + 1*rho_2^0.5
            'ndensity': 1,
            'rho_core_cut': 2000,
            'drho_core_cut': 250
            },

        'Mo': {
            'npot': 'FinnisSinclairShiftedScaled', ## linear embedding function: 1*rho_1^1
            'fs_parameters': [1, 1],
            'ndensity': 1,
            'rho_core_cut': 3000,
            'drho_core_cut': 150
            }
        },
    'functions':
    {
                        'Fe':  {
                                'nradmax_by_orders': [15, 3, 1, 1, 1],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                },
                        'Mo':  {
                                'nradmax_by_orders': [15, 4, 1, 1, 1],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                },
                        'BINARY':  {
                                'nradmax_by_orders': [15, 5, 3, 2, 1],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                }
       }
}
)

In [ ]:
NewFuncsACEer = MyPyACECalculator(multispace_basis_config=new_multispace_config ) # components = components,

In [ ]:
NewFuncsACEer.get_ace_projections(AtomsObjects['atoms'].iloc[2])

In [ ]:
NewFuncsACEFEATURES = AtomsObjects['atoms'].map(NewFuncsACEer.get_ace_projections)

In [ ]:
from pyace import 

In [ ]:
NewFuncsACEFEATURES.name = 'ace_projections_specific_1'

In [ ]:
expand_newfuncs_ace = gf.array_expansions(NewFuncsACEFEATURES.to_frame(),['ace_projections_specific_1']) 

In [ ]:
CNAV_NEWF_ACE = gf.featurize_dataframe(expand_newfuncs_ace, CNList)

In [ ]:
NEWF_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NEWF_ACE-CNAV.pkl')

In [ ]:
from Tools.DatasetTools.ACEDescriptors import MyPyACECalculator
TestMyPyAce = SourceFileLoader('TestMyPyAce', 'Tools/DatasetTools/ACEDescriptors.py').load_module().TestMyPyAce
Test = TestMyPyAce()
Test.setUpClass()

In [ ]:
new_multispace_config['embeddings'] = {
        'Fe': {
            'npot': 'FinnisSinclairShiftedScaled',
            'fs_parameters': [1, 1], ## non-linear embedding function: 1*rho_1^1 + 1*rho_2^0.5
            'ndensity': 1,
            'rho_core_cut': 2000,
            'drho_core_cut': 250
            },

        'Mo': {
            'npot': 'FinnisSinclairShiftedScaled', ## linear embedding function: 1*rho_1^1
            'fs_parameters': [1, 1],
            'ndensity': 1,
            'rho_core_cut': 3000,
            'drho_core_cut': 150
            }
        }
new_multispace_config['functions'] = {
                        'Fe':  {
                                'nradmax_by_orders': [15, 4, 2, 2, 2],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                },
                        'Mo':  {
                                'nradmax_by_orders': [15, 5, 3, 1, 1],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                },
                        'BINARY':  {
                                'nradmax_by_orders': [5, 4, 3, 1, 1],
                                'lmax_by_orders': [ 0, 3, 2,1,0],
                                }
       }

In [ ]:
fig, ax = Test.test_unary_specific_functions(_new_multispace_config=new_multispace_config)

<div style='background:#999966'>
    <li> not much effect on triplets param 
    <li> not much effect on lmax params 
</div>

In [ ]:
new_multispace_config = copy.deepcopy(Test.default_multispace_config)
new_multispace_config['embeddings'] = {                                                                                  
     'Fe': {
         'npot': 'FinnisSinclairShiftedScaled',
         'fs_parameters': [1, 1], ## non-linear embedding function: 1*rho_1^1 + 1*rho_2^0.5
         'ndensity': 1,
         'rho_core_cut': 2000,
         'drho_core_cut': 250
         }, 

     'Mo': {
         'npot': 'FinnisSinclairShiftedScaled', ## linear embedding function: 1*rho_1^1
         'fs_parameters': [1, 1],
         'ndensity': 1,
         'rho_core_cut': 3000,
         'drho_core_cut': 150
         }
     }
new_multispace_config['functions'] = {
                     'UNARY':  {
                             'nradmax_by_orders': [5, 1, 1, 1, 1],
                             'lmax_by_orders': [ 0, 3, 2,1,0],
                             },
#                      'Mo':  {
#                              'nradmax_by_orders': [5, 1, 1, 1, 1],
#                              'lmax_by_orders': [ 0, 3, 2,1,0],
#                              },
                     'BINARY':  {
                             'nradmax_by_orders': [5, 1, 1, 1, 1],
                             'lmax_by_orders': [ 0, 3, 2,1,0],
                             }
    }

In [ ]:
# new_multispace_config['functions'] = {
#                      'ALL':  {
#                              'nradmax_by_orders': [5, 1, 1, 1, 1],
#                              'lmax_by_orders': [ 0, 3, 2,1,0],
#                              }
#     }

In [ ]:
bas= construct_bbasisconfiguration(new_multispace_config)

In [ ]:
bas.funcspecs_blocks[2].funcspecs

In [ ]:
bas.funcspecs_blocks[2].funcspecs

In [ ]:
construct_bbasisconfiguration??

In [ ]:
Test.default_multispace_config

In [ ]:
construct_bbasisconfiguration(Test.default_multispace_config)

In [ ]:
ShorterFunctionsOptions = copy.deepcopy(default_options)

In [ ]:
ShorterFunctionsOptions['functions'] = new_functions

In [ ]:
ShorterFunctionsACE = MyPyACECalculator(components=['Fe','Mo'], multispace_basis_config=ShorterFunctionsOptions)

In [ ]:
ACEFEATURES_shorter_functions = AtomsObjects['atoms'].map(ShorterFunctionsACE.get_ace_projections)
ACEFEATURES.name = 'ace_projections_shorter_functions'

# SOAPFeatures

In [ ]:
from dscribe.descriptors import SOAP
from mendeleev import element
import ase
from sklearn.feature_selection import VarianceThreshold

In [ ]:
atomsobjectlocation = os.path.join(dataset, 'Atomsobjects')
atomsobjectfile = os.path.join(atomsobjectlocation,f'{system}-POSCAR-initial-rescaled-AtomsObjects.pkl')

In [ ]:
AtomsObjects = pd.read_pickle(atomsobjectfile).dropna()

In [ ]:
soap_params = dict(rcut = 4, nmax = 9, lmax = 6, sigma = 0.1, rbf = 'gto', periodic = True, crossover = True)
params_str = '__'.join([f'{key}_{val}' for key, val in soap_params.items()])

## Canonical

In [ ]:
def reset_symbols(a: ase.atoms.Atoms, newsym : str = 'W'):
    newa = a.copy()
    natoms = newa.get_global_number_of_atoms()
    newsymbols = [newsym]*natoms
    newa.set_chemical_symbols(newsymbols)
    return newa

In [ ]:
soapcases = ['canonicalFe','canonicalW', 'specific']

In [ ]:
SOAPFEATURES = {}
EXPANDED_SOAP = {}
AVE_SOAP = {}
variances = {}
SEL_SOAP = {}
FINAL_SOAP = {}
soap_params = dict(rcut = 4, nmax = 9, lmax = 6, sigma = 0.1, rbf = 'gto', periodic = True, crossover = True)
params_str = '__'.join([f'{key}_{val}' for key, val in soap_params.items()])
soap_features_file={}

In [ ]:
for soapcase in soapcases:
    print(soapcase)
    soap_features_file.update({soapcase: os.path.join(descriptorslocation, f'soap_features__{soapcase}__{params_str}.kpl')})
    if 'canonicalFe' in soapcase:
        species=[26]
        thisatomsobjects = AtomsObjects['atoms'].map(lambda a: reset_symbols(a, 'Fe'))
    elif 'canonicalW' in soapcase:
        species=[74]
        thisatomsobjects = AtomsObjects['atoms'].map(lambda a: reset_symbols(a, 'W'))
    elif 'specific' in soapcase:
        species = [element(s).atomic_number for s in dataset.split('-')]
        thisatomsobjects = AtomsObjects['atoms'].map(lambda a: copy.deepcopy(a))
    SOAPER = SOAP(species=species, **soap_params)
    SOAPFEATURES.update({soapcase: thisatomsobjects.map(SOAPER.create)}) #,pd.DataFrame(data= columns=['SOAP'])
    EXPANDED_SOAP.update({soapcase: gf.array_expansions(SOAPFEATURES[soapcase].to_frame(name='SOAP'), ['SOAP'])})
    AVE_SOAP.update({soapcase: gf.featurize_dataframe(EXPANDED_SOAP[soapcase], CNList)})
    variances.update({soapcase: {name: col.var() for name, col in AVE_SOAP[soapcase].iteritems()}})
    selector = VarianceThreshold(threshold=1e-7)
    SEL_SOAP.update({soapcase: selector.fit_transform(AVE_SOAP[soapcase])})
    FINAL_SOAP.update({soapcase: pd.DataFrame(data=SEL_SOAP[soapcase], columns = selector.get_feature_names_out(), index=AVE_SOAP[soapcase].index)})
    FINAL_SOAP[soapcase].to_pickle(soap_features_file[soapcase])

In [ ]:
soap_features_file

# Pyscal features 

In [ ]:
atomsobjectlocation = os.path.join(dataset, 'Atomsobjects')
atomsobjectfile = os.path.join(atomsobjectlocation,f'{system}-POSCAR-initial-rescaled-AtomsObjects.pkl')

In [ ]:
from tqdm.auto import tqdm
from Tools.DatasetTools import pyscalfeaturizers as pf
from BopFoxFeaturizer.struct_db import struct_db
AtomsObjects = pd.read_pickle(atomsobjectfile).dropna()

##  Coordination Numbers

In [ ]:
featurizers = [pf.pyscal_steinhardt, pf.pyscal_cn] #, get_steinhardt]
pyscal_features = [feature.__name__ for feature in featurizers]

pyscalsteinhardt = os.path.join(descriptorslocation, 'pyscal_steinhardt.kpl')

if os.path.exists(pyscalsteinhardt):
    PyscalFeatures = pd.read_pickle(pyscalsteinhardt)
else:
    PyscalFeatures = pf.featurize_many(AtomsObjects,  featurizers, colid='atoms')
    expanded_ste = pf.expand_features(PyscalFeatures.pyscal_steinhardt, 'pyscal_steinhardt')
    PyscalFeatures = pd.concat([expanded_ste, PyscalFeatures.pyscal_cn], axis=1)
    PyscalFeatures.to_pickle(pyscalsteinhardt)

In [ ]:
PyscalFeatures

## Stainhardt parameters 

From the Steinhardt parameters obtained by Pyscal library, we also want to average over the coordination polyhedra. This time we are also saving the total average for each parameter.

$$ q_{j, CP} ^i = \dfrac{1}{n_{at}^i}\sum _{at \in CP} q_{j, at} ^i $$

In [ ]:
thisFeatures = PyscalFeatures[['pyscal_steinhardt_0','pyscal_steinhardt_1']]

In [ ]:
intersect = thisFeatures.index.intersection(CNList.index)

In [ ]:
CNPyscal  = gf.featurize_many(thisFeatures.loc[intersect], CNList[intersect], [gf.cn_average])

In [ ]:
CNPyscal

In [ ]:
PyscalFeaturesFile = os.path.join(descriptorslocation,'CNAVPyscal.pkl')

In [ ]:
CNPyscal.to_pickle(PyscalFeaturesFile)

# Characterization of Features 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.pairplot(CN)

##  Correlations

In [ ]:
plt.rc('font',size=22)

In [ ]:
BS['EF'] = BS['EF'].astype(float)

In [ ]:
FeatureGroups = {'density features': DensitiFeatures, 'atomic features': AtomicFeaturesMagpie, 'composition features': CompositionFeatures, 'Dataset Features': DatasetFeatures}
TargetCorrelations = {groupname: GroupFeatures.corrwith(BS['EF']).abs().dropna().sort_values(ascending=False) for groupname, GroupFeatures in FeatureGroups.items()}

In [ ]:
len(TargetCorrelations)

In [ ]:
fig = plt.figure(figsize=(len(TargetCorrelations)*7, 10))
border=0
totalfeatures=18
for i, (group, TargetCorr) in enumerate(TargetCorrelations.items()):
    nfeatures = len(TargetCorr[:5])
    ax = fig.add_axes([border/totalfeatures,0.2,(nfeatures)/totalfeatures,0.7])
    border +=nfeatures
    ax.bar( TargetCorr[:5].index,TargetCorr[:5].values) #, ax = ax, orient='vertical')
axes = fig.get_axes()
[tax.set_yticklabels(tax.get_yticklabels(), visible=False) for tax in axes[1:]]
[tax.sharey(axes[0]) for tax in axes[:1]]
#axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90) 

# Plot correlations for most correlated

In [ ]:
for name, item in DensitiFeatures.iteritems():
    print(name)

In [ ]:
DensitiFeatures[DensitiFeatures.vpa < 6]

In [ ]:
sns.histplot(DensitiFeatures.vpa)

In [ ]:
fig, axes = plt.subplots(1, 3, sharey = True, figsize=(15,5))
for (fname, feature), ax in zip(DensitiFeatures.iteritems(), axes):
    feature.hist(ax=ax)
#    sns.histplot(feature, ax =ax)

## By hand outlier detection:

In [ ]:
selection = (FeatureGroups['density features']['packing fraction'] < 3) & (FeatureGroups['density features']['vpa']>8) &(FeatureGroups['density features']['density']<75)

In [ ]:
def target_correlation_scatters(thisgroup, selection=None):
    featurenames = TargetCorrelations[thisgroup].index.to_list()
    if selection is None:
        selection = FeatureGroups[thisgroup].index
    nplots =  min([4, len(featurenames)])
    fig, axes = plt.subplots(1, nplots,  figsize=(7*4, 10), sharey=True)
    intersect = selection.intersection(BS['EF'].index)
    for ax, thisfeature in zip(axes, featurenames[:nplots]):
        ax.scatter(FeatureGroups[thisgroup][thisfeature][intersect], BS['EF'][intersect])
        ax.set_xlabel(thisfeature)
    axes[0].set_ylabel('$\Delta E_f$')

In [ ]:
target_correlation_scatters('atomic features')

In [ ]:
target_correlation_scatters('density features')#, selection=selection)

In [ ]:
target_correlation_scatters('composition features')

In [ ]:
target_correlation_scatters('composition features')

In [ ]:
target_correlation_scatters('Dataset Features')

In [ ]:
TargetCorrelations.keys()